# Analysis on Spread of Asset Pairs

This notebook covers the analysis of the spread of pre-selected asset pairs determined by sector, where it will obtain the historical prices of the ticker data in the out-of-sample range between January 1, 2019 to December 31, 2023.

Next, spread analysis will be performed where the past prices of each equity is used and tested if the pre-selected asset pairs were cointegrated by running Engle-Granger tests where cointegrated asset pairs are assessed by having a p-score of less than 0.5.

For each cointegrated asset pairs, the spread between each asset pair is calculated, alongside their characterisation, distributions, spread levels, rolling half-life and rolling z-scores based on thresholds.

It will then save the computed results into CSVs stored on `../../data/processed/` where the next notebook `02_cointegration_validation` will test if the cointegration holds for in-sample price data.

# Imports

In [1]:
import sys
import os
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data import get_close_prices, get_market_cap, get_ohlcv
from src.modelling import TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END, INTERVAL, TICKER_NAMES, CANDIDATE_PAIRS
from src.modelling import engle_granger_test, screen_pairs, adf_test # co-integration
from src.modelling import spread_summary, compute_spread, compute_zscore, compute_rolling_half_life, compute_rolling_zscore # spread analysis


# Getting Historical Prices

For cointegration-based spread analysis, we study stocks with strong economic 
relationships within the same sector. Same-sector pairs share common macroeconomic 
drivers — such as interest rates, commodity prices, and regulation — making them 
strong candidates for cointegration.

We focus on the following sectors and candidate pairs:

| Sector | Pair | Tickers |
|---|---|---|
| Technology (Semiconductors) | Nvidia / AMD | `NVDA.O` / `AMD.O` |
| Technology (Semiconductors) | Nvidia / TSMC | `NVDA.O` / `TSM.N` |
| Consumer Staples (Beverages) | Coca-Cola / PepsiCo | `KO.N` / `PEP.O` |
| Financials (Banking) | JPMorgan / Bank of America | `JPM.N` / `BAC.N` |
| Financials (Banking) | Goldman Sachs / Morgan Stanley | `GS.N` / `MS.N` |
| Energy | ExxonMobil / Chevron | `XOM.N` / `CVX.N` |
| E-Commerce / Cloud | Amazon / Microsoft | `AMZN.O` / `MSFT.O` |
| E-Commerce / Cloud | Meta / Alphabet | `META.O` / `GOOGL.O` |
| Healthcare Pharma | Johnson & Johnson / Pfizer | `JNJ.O` / `PFE.O` |


We retrieve daily closing prices for all tickers over the full date range, 
then split into in-sample (training) and out-of-sample (test) sets for 
validation:

- **Training period**: 1 January 2020 – 31 December 2023
- **Test period**: 1 January 2024 – 31 December 2025


## Get Market Cap
Obtain market cap prices as of 29th December 2023 (last trading day of 2023 - end of test period)

In [2]:
market_cap_df = get_market_cap(
    TICKERS,
    start="2023-12-29",
    end="2023-12-29",
)

market_cap_df

[cache hit] AMD-O_AMZN-O_BAC-N_CVX-N_GOOGL-O_GS-N_JN__1D__4c761d2e99.csv


,NVDA.O,AMD.O,TSM.N,KO.N,PEP.O,JPM.N,BAC.N,GS.N,MS.N,XOM.N,CVX.N,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N
Date,,,,,,,,,,,,,,,,,
2023-12-29,1223193400000,2.381407e+11,5.011967e+11,2.547788e+11,2.335069e+11,4.917605e+11,2.664554e+11,1.258044e+11,153052304835,3.995975e+11,2.807263e+11,1.570153e+12,2.794828e+12,9.096286e+11,1755459040000,3.773169e+11,1.625602e+11


## Obtain ticker prices

Obtain ticker prices for all TICKER symbols from start of training period (2019-01-01) to end of training period (2023-12-31) date with interval of 1 day

In [3]:
prices_df = get_close_prices(
    TICKERS,
    start=TRAIN_START,
    end=TRAIN_END,
    interval=INTERVAL
)

[cache partial] Fetching 2019-01-01 → 2019-01-01
LSEG session opened.
[cache partial] Skipping (no trading data): LSEG returned no data for ['NVDA.O', 'AMD.O', 'TSM.N', 'KO.N', 'PEP.O', 'JPM.N', 'BAC.N', 'GS.N', 'MS.N', 'XOM.N', 'CVX.N', 'AMZN.O', 'MSFT.O', 'META.O', 'GOOGL.O', 'JNJ.N', 'PFE.N'] (2019-01-01 to 2019-01-01)
[cache partial] Fetching 2023-12-30 → 2023-12-31
[cache partial] Skipping (no trading data): LSEG returned no data for ['NVDA.O', 'AMD.O', 'TSM.N', 'KO.N', 'PEP.O', 'JPM.N', 'BAC.N', 'GS.N', 'MS.N', 'XOM.N', 'CVX.N', 'AMZN.O', 'MSFT.O', 'META.O', 'GOOGL.O', 'JNJ.N', 'PFE.N'] (2023-12-30 to 2023-12-31)


In [4]:
print(prices_df.shape)
prices_df.head()

(1258, 17)


,NVDA.O,AMD.O,TSM.N,KO.N,PEP.O,JPM.N,BAC.N,GS.N,MS.N,XOM.N,CVX.N,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N
Date,,,,,,,,,,,,,,,,,
2019-01-02,3.40550,18.83,36.52,46.93,109.28,99.31,24.96,172.03,40.40,69.69,110.69,76.9565,101.12,135.68,52.7340,127.75,40.998794
2019-01-03,3.19975,17.05,34.36,46.64,108.26,97.11,24.56,169.51,39.68,68.62,108.57,75.0140,97.40,131.74,51.2735,125.72,39.851776
2019-01-04,3.40475,19.00,34.97,47.57,110.48,100.69,25.58,175.05,41.30,71.15,110.82,78.7695,101.93,137.95,53.9035,127.83,40.761807
2019-01-07,3.58500,20.57,35.23,46.95,109.53,100.76,25.56,176.02,41.71,71.52,112.26,81.4755,102.06,138.05,53.7960,127.01,40.979835
2019-01-08,3.49575,20.75,34.94,47.48,110.58,100.57,25.51,175.37,41.45,72.04,111.77,82.8290,102.80,142.53,54.2685,129.96,41.169425


### Example

Plotting the closing price of NVDA.O from January 1, 2019 to December 31, 2023

In [5]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prices_df.index,
    y=prices_df['NVDA.O'],
    mode='lines',
    name='NVDA.O',
    line=dict(color='#76b900', width=1.5)
))

fig.update_layout(
    title='NVDA.O Close Price (Jan 2019 – Dec 2023)',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    xaxis_rangeslider_visible=False,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

# Spread Analysis

## Obtain Pre-Selected Candidate Pairs

In [6]:
CANDIDATE_PAIRS

[('NVDA.O', 'AMD.O'),
 ('NVDA.O', 'TSM.N'),
 ('KO.N', 'PEP.O'),
 ('JPM.N', 'BAC.N'),
 ('GS.N', 'MS.N'),
 ('XOM.N', 'CVX.N'),
 ('AMZN.O', 'MSFT.O'),
 ('META.O', 'GOOGL.O'),
 ('JNJ.N', 'PFE.N')]

## Run Engle-Granger Cointegration Tests on Candidate Pairs

Convert raw closing prices to log prices

In [7]:
prices_log = np.log(prices_df)

In [8]:
results_df = screen_pairs(prices_log, CANDIDATE_PAIRS, significance=0.05)

In [9]:
results_df

,y,x,hedge_ratio,intercept,adf_stat,p_value,is_cointegrated
0,CVX.N,XOM.N,0.713964,1.761593,-3.462651,0.009000,True
1,KO.N,PEP.O,0.664058,0.674101,-3.313506,0.014285,True
2,GS.N,MS.N,0.800362,2.256544,-3.252029,0.017164,True
3,AMD.O,NVDA.O,0.630470,2.596369,-2.769685,0.062736,False
4,JNJ.N,PFE.N,0.367091,3.691649,-2.675913,0.078300,False
5,AMZN.O,MSFT.O,0.500083,2.112186,-1.651058,0.456489,False
6,META.O,GOOGL.O,0.538279,2.974854,-1.508262,0.529421,False
7,TSM.N,NVDA.O,0.418804,3.261647,-1.433971,0.565886,False
8,BAC.N,JPM.N,0.879908,-0.798858,-0.982241,0.759686,False


### Filter Results

In [10]:
cointegrated_pairs = results_df[results_df['p_value'] < 0.05].copy()
print(cointegrated_pairs[['y', 'x', 'hedge_ratio', 'p_value']])


       y      x  hedge_ratio   p_value
0  CVX.N  XOM.N     0.713964  0.009000
1   KO.N  PEP.O     0.664058  0.014285
2   GS.N   MS.N     0.800362  0.017164


### Compute spreads

In [11]:
summaries = []
for _, row in cointegrated_pairs.iterrows():
    y = prices_log[row['y']]
    x = prices_log[row['x']]
    
    summary = spread_summary(
        y, x,
        hedge_ratio=row['hedge_ratio'],
        intercept=row['intercept'],
        window=60
    )
    summary['pair'] = f"{row['y']}/{row['x']}"
    summaries.append(summary)

summary_df = pd.DataFrame(summaries).set_index('pair')
summary_df


,half_life,hurst,current_zscore,spread_mean,spread_std,adf_stat,adf_pvalue,is_stationary
pair,,,,,,,,
CVX.N/XOM.N,33.850675,0.336406,0.628101,-4.021514e-15,0.062838,-3.462651,0.009000,True
KO.N/PEP.O,40.887317,0.418551,0.798284,-1.135710e-14,0.049604,-3.313506,0.014285,True
GS.N/MS.N,36.356403,0.397755,0.592609,5.946136e-15,0.051924,-3.252029,0.017164,True


## Visualise Spreads

### Visualise co-movements

In [12]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']} vs {row['x']} — Normalised Log Prices"
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    y = prices_log[row['y']]
    x = prices_log[row['x']]
    
    # Normalise to 0 at start
    y_norm = y - y.iloc[0]
    x_norm = x - x.iloc[0]

    fig.add_trace(go.Scatter(
        x=y_norm.index, y=y_norm.values,
        name=row['y'], line=dict(width=1.5)
    ), row=i, col=1)
    
    fig.add_trace(go.Scatter(
        x=x_norm.index, y=x_norm.values,
        name=row['x'], line=dict(width=1.5, dash='dash')
    ), row=i, col=1)

fig.update_layout(
    height=900,
    title_text="Normalised Log Price Series — Cointegrated Pairs (Jan 2019 – Dec 2023)",
    showlegend=True
)
fig.show()

#### Load to matplotlib

In [13]:
import matplotlib.dates as mdates
import scienceplots

plt.style.use(['science', 'high-contrast'])
plt.rcParams['text.usetex'] = True

fig, axes = plt.subplots(
    nrows=len(cointegrated_pairs),
    ncols=1,
    figsize=(10, 9),
    sharex=True
)
if len(cointegrated_pairs) == 1:
    axes = [axes]

panel_labels = [f"({chr(97 + i)})" for i in range(len(cointegrated_pairs))]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    y = prices_log[row["y"]]
    x = prices_log[row["x"]]
    y_norm = y - y.iloc[0]
    x_norm = x - x.iloc[0]
    ax.plot(y_norm.index, y_norm.values, label=row["y"], linewidth=1.5)
    ax.plot(x_norm.index, x_norm.values, label=row["x"], linestyle="--", linewidth=1.5)
    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")
    ax.set_ylabel("Log price (normalised)")
    ax.legend(loc="upper left", ncol=2, fontsize=9)  # legend on every subplot

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.tight_layout()
fig.savefig("figures/normalised_prices.pdf", bbox_inches="tight")
plt.close(fig)

### Spread Characterisation

In [14]:
from plotly.subplots import make_subplots

pairs      = summary_df.index.tolist()
half_lives = summary_df['half_life'].tolist()
hursts     = summary_df['hurst'].tolist()
zscores    = summary_df['current_zscore'].tolist()
adf_pvals  = summary_df['adf_pvalue'].tolist()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Half-Life of Mean Reversion (Trading Days)",
        "Hurst Exponent",
        "Current Z-Score (End of Training Window)",
        "ADF p-value on Spread"
    )
)

# --- Half-life bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=half_lives,
    marker_color=['green' if h < 60 else 'orange' for h in half_lives],
    text=[f"{h:.1f}d" for h in half_lives], textposition='auto',
    name='Half-Life'
), row=1, col=1)
fig.add_hline(y=60, line=dict(color='red', dash='dash'), annotation_text='60d threshold', row=1, col=1)

# --- Hurst exponent bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=hursts,
    marker_color=['green' if h < 0.5 else 'red' for h in hursts],
    text=[f"{h:.3f}" for h in hursts], textposition='auto',
    name='Hurst'
), row=1, col=2)
fig.add_hline(y=0.5, line=dict(color='red',    dash='dash'), annotation_text='Random walk (0.5)', row=1, col=2)

# --- Current z-score bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=zscores,
    marker_color=['red' if abs(z) > 2 else 'steelblue' for z in zscores],
    text=[f"{z:.2f}" for z in zscores], textposition='auto',
    name='Z-Score'
), row=2, col=1)
fig.add_hline(y=2,  line=dict(color='orange', dash='dot'), row=2, col=1)
fig.add_hline(y=-2, line=dict(color='orange', dash='dot'), row=2, col=1)
fig.add_hline(y=0,  line=dict(color='gray',   dash='dash'), row=2, col=1)

# --- ADF p-value bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=adf_pvals,
    marker_color=['green' if p < 0.05 else 'red' for p in adf_pvals],
    text=[f"{p:.3f}" for p in adf_pvals], textposition='auto',
    name='ADF p-value'
), row=2, col=2)
fig.add_hline(y=0.05, line=dict(color='red', dash='dash'), annotation_text='5% significance', row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Spread Characterisation Summary — Cointegrated Pairs",
    showlegend=False
)

fig.show()

#### add to markdown

In [15]:
import matplotlib.patches as mpatches

pairs      = summary_df.index.tolist()
half_lives = summary_df["half_life"].tolist()
hursts     = summary_df["hurst"].tolist()
zscores    = summary_df["current_zscore"].tolist()
adf_pvals  = summary_df["adf_pvalue"].tolist()

PASS  = dict(color="white", edgecolor="black", linewidth=0.8)
FAIL  = dict(color="white", edgecolor="black", linewidth=0.8, hatch="///")
REF   = dict(color="black", linestyle="--", linewidth=1)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
ax_hl, ax_hu, ax_z, ax_adf = axes.flatten()

# Half-life
for x, (pair, h) in enumerate(zip(pairs, half_lives)):
    ax_hl.bar(x, h, **(PASS if h < 60 else FAIL))
    ax_hl.text(x, h, f"{h:.1f}d", ha="center", va="bottom", fontsize=9)
ax_hl.set_xticks(range(len(pairs))); ax_hl.set_xticklabels(pairs)
ax_hl.axhline(60, **REF)
ax_hl.text(len(pairs)-1, 60, "60d threshold", ha="right", va="bottom")
ax_hl.set_ylabel("Days")

# Hurst
for x, (pair, h) in enumerate(zip(pairs, hursts)):
    ax_hu.bar(x, h, **(PASS if h < 0.5 else FAIL))
    ax_hu.text(x, h, f"{h:.3f}", ha="center", va="bottom", fontsize=9)
ax_hu.set_xticks(range(len(pairs))); ax_hu.set_xticklabels(pairs)
ax_hu.axhline(0.5, **REF)
ax_hu.text(len(pairs)-1, 0.5, "Random walk (0.5)", ha="right", va="bottom")
ax_hu.set_ylabel("H")

# Z-score
for x, (pair, z) in enumerate(zip(pairs, zscores)):
    ax_z.bar(x, z, **(FAIL if abs(z) > 2 else PASS))
    ax_z.text(x, z, f"{z:.2f}", ha="center", va=("bottom" if z >= 0 else "top"), fontsize=9)
ax_z.set_xticks(range(len(pairs))); ax_z.set_xticklabels(pairs)
ax_z.axhline(0,  color="black", linestyle="--", linewidth=1)
ax_z.axhline( 2, color="black", linestyle=":",  linewidth=1)
ax_z.axhline(-2, color="black", linestyle=":",  linewidth=1)
ax_z.set_ylabel("Z-score")

# ADF p-value
for x, (pair, p) in enumerate(zip(pairs, adf_pvals)):
    ax_adf.bar(x, p, **(PASS if p < 0.05 else FAIL))
    ax_adf.text(x, p, f"{p:.3f}", ha="center", va="bottom", fontsize=9)
ax_adf.set_xticks(range(len(pairs))); ax_adf.set_xticklabels(pairs)
ax_adf.axhline(0.05, **REF)
ax_adf.text(len(pairs)-1, 0.05, r"5\% significance", ha="right", va="bottom")
ax_adf.set_ylabel("p-value")

# Panel labels (a)–(d) instead of titles
for ax, label in zip(axes.flatten(), ["(a)", "(b)", "(c)", "(d)"]):
    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")

# Shared legend
pass_patch = mpatches.Patch(facecolor="white", edgecolor="black", label="Pass threshold")
fail_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="///", label="Fail threshold")
fig.legend(handles=[pass_patch, fail_patch], loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))

for ax in axes.flatten():
    ax.tick_params(axis="x", rotation=30)

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/spread_characterisation.pdf", bbox_inches="tight")
plt.close(fig)

### Spread level between +1 std to +2 std

In [16]:
fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']}/{row['x']} — Spread Level" 
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']], 
                           row['hedge_ratio'], row['intercept'])
    
    # Spread line
    fig.add_trace(go.Scatter(
        x=spread.index, y=spread.values, 
        line=dict(width=1, color='steelblue'), 
        name='Spread'), row=i, col=1)

    # Mean
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean()] * len(spread),
        line=dict(color='red', dash='dash', width=1),
        name='Mean', showlegend=(i == 1)), row=i, col=1)

    # ±1σ
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() + spread.std()] * len(spread),
        line=dict(color='orange', dash='dot', width=1),
        name='+1σ', showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() - spread.std()] * len(spread),
        line=dict(color='orange', dash='dot', width=1),
        name='-1σ', showlegend=(i == 1)), row=i, col=1)

    # ±2σ
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() + 2*spread.std()] * len(spread),
        line=dict(color='red', dash='dot', width=1),
        name='+2σ', showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() - 2*spread.std()] * len(spread),
        line=dict(color='red', dash='dot', width=1),
        name='-2σ', showlegend=(i == 1)), row=i, col=1)

fig.update_layout(
    height=900, 
    title_text="Spread Levels with σ Bands", 
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

#### Save as matplotlib

In [17]:
import matplotlib.dates as mdates
import scienceplots

plt.style.use(['science', 'high-contrast'])
plt.rcParams['text.usetex'] = True

n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(10, 9), sharex=True)
if n_pairs == 1:
    axes = [axes]

panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]

# Only label on the first axis, suppress on the rest
for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    mu    = spread.mean()
    sigma = spread.std(ddof=1)

    first = (label == panel_labels[0])
    ax.plot(spread.index, spread.values, linewidth=1, label="Spread" if first else None)
    ax.axhline(mu,           linestyle="--", linewidth=1,   label="Mean" if first else None)
    ax.axhline(mu + sigma,   linestyle=":",  linewidth=0.8, label=r"$\pm1\sigma$" if first else None)
    ax.axhline(mu - sigma,   linestyle=":",  linewidth=0.8)
    ax.axhline(mu + 2*sigma, linestyle="-.", linewidth=0.8, label=r"$\pm2\sigma$" if first else None)
    ax.axhline(mu - 2*sigma, linestyle="-.", linewidth=0.8)
    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")
    ax.set_ylabel("Spread")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Single shared legend outside, below the figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/spread_levels.pdf", bbox_inches="tight")
plt.close(fig)

### Z-scores visualisation

In [18]:
fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']}/{row['x']} — Rolling Z-Score (60d)"
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']],
                           row['hedge_ratio'], row['intercept'])
    zscore = compute_zscore(spread, window=60)

    fig.add_trace(go.Scatter(x=zscore.index, y=zscore.values,
                             line=dict(width=1, color='purple')), row=i, col=1)
    fig.add_hline(y=0,  line=dict(color='red',    dash='dash'), row=i, col=1)
    fig.add_hline(y=2,  line=dict(color='orange', dash='dot'),  row=i, col=1)
    fig.add_hline(y=-2, line=dict(color='orange', dash='dot'),  row=i, col=1)

fig.update_layout(height=900, title_text="Rolling Z-Scores", showlegend=False)
fig.show()


#### Save to Matplotlib

In [19]:
import matplotlib.dates as mdates
import scienceplots

plt.style.use(['science', 'high-contrast'])
plt.rcParams['text.usetex'] = True

window = 60
n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(10, 9), sharex=True)
if n_pairs == 1:
    axes = [axes]

panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]],
        prices_log[row["x"]],
        row["hedge_ratio"],
        row["intercept"],
    ).dropna()
    zscore = compute_zscore(spread, window=window).dropna()

    first = (label == panel_labels[0])
    ax.plot(zscore.index, zscore.values, linewidth=1, label="Z-score" if first else None)
    ax.axhline(0,  linestyle="--", linewidth=1,   label="Mean" if first else None)
    ax.axhline( 2, linestyle=":",  linewidth=0.8, label=r"$\pm2\sigma$" if first else None)
    ax.axhline(-2, linestyle=":",  linewidth=0.8)

    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")
    ax.set_ylabel("Z-score")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/spread_zscores.pdf", bbox_inches="tight")
plt.close(fig)

### Spread Distribution

In [20]:
from scipy import stats

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    f"{row['y']}/{row['x']}" for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']],
                           row['hedge_ratio'], row['intercept']).dropna()

    # Histogram
    fig.add_trace(go.Histogram(x=spread.values, nbinsx=50, 
                               histnorm='probability density',
                               marker_color='steelblue', opacity=0.7,
                               name='Spread'), row=1, col=i)

    # Normal overlay
    x_range = np.linspace(spread.min(), spread.max(), 200)
    normal_pdf = stats.norm.pdf(x_range, spread.mean(), spread.std())
    fig.add_trace(go.Scatter(x=x_range, y=normal_pdf,
                             line=dict(color='red', width=2),
                             name='Normal fit'), row=1, col=i)

fig.update_layout(height=400, title_text="Spread Distributions", showlegend=False)
fig.show()


#### Save to Matplotlib

In [21]:
from scipy import stats
import scienceplots

plt.style.use(['science', 'high-contrast'])
plt.rcParams['text.usetex'] = True

n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=1, ncols=n_pairs, figsize=(12, 4), sharey=True)
if n_pairs == 1:
    axes = [axes]

panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]],
        prices_log[row["x"]],
        row["hedge_ratio"],
        row["intercept"],
    ).dropna()

    x_range    = np.linspace(spread.min(), spread.max(), 200)
    normal_pdf = stats.norm.pdf(x_range, spread.mean(), spread.std())

    first = (label == panel_labels[0])
    ax.hist(spread.values, bins=50, density=True, alpha=0.6,
            label="Spread" if first else None)
    ax.plot(x_range, normal_pdf, linewidth=1.5,
            label="Normal fit" if first else None)

    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")
    ax.set_xlabel("Spread")

axes[0].set_ylabel("Density")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig("figures/spread_distributions.pdf", bbox_inches="tight")
plt.close(fig)

### Rolling Half-Life

In [22]:
fig = make_subplots(
    rows=len(cointegrated_pairs), cols=1,
    subplot_titles=[
        f"{row['y']}/{row['x']} Rolling Half-Life"
        for _, row in cointegrated_pairs.iterrows()
    ],
    shared_xaxes=True
)

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()

    rhl        = compute_rolling_half_life(spread).replace(np.inf, np.nan)
    rhl_smooth = rhl.ewm(span=21, adjust=False).mean()

    # Raw (faint background)
    fig.add_trace(go.Scatter(
        x=rhl.index, y=rhl.values,
        mode="lines",
        line=dict(width=0.8, color="steelblue"),
        opacity=0.3,
        name="Raw", showlegend=(i == 1)
    ), row=i, col=1)

    # EWM smoothed (prominent)
    fig.add_trace(go.Scatter(
        x=rhl_smooth.index, y=rhl_smooth.values,
        mode="lines",
        line=dict(width=1.8, color="steelblue"),
        name="EWM (21d)", showlegend=(i == 1)
    ), row=i, col=1)

    fig.add_hline(
        y=60, line=dict(color="red", dash="dash", width=1),
        annotation_text="60d threshold", annotation_position="top right",
        row=i, col=1
    )

fig.update_layout(
    height=300 * len(cointegrated_pairs),
    title_text="Rolling Half-Life (252-day window, EWM-smoothed) — Cointegrated Pairs",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.update_yaxes(title_text="Half-Life (days)", rangemode="tozero")
fig.show()


In [23]:
n = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(10, 3 * n), sharex=True)

if n == 1:
    axes = [axes]

panel_labels = [f"({chr(97 + i)})" for i in range(n)]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()

    rhl        = compute_rolling_half_life(spread, window=252).replace(np.inf, np.nan)
    rhl_smooth = rhl.ewm(span=21, adjust=False).mean()

    # Raw (faint background)
    ax.plot(rhl.index, rhl.values,
            linewidth=0.8, alpha=0.3,
            label="Raw" if label == panel_labels[0] else None)

    # EWM smoothed (prominent)
    ax.plot(rhl_smooth.index, rhl_smooth.values,
            linewidth=1.8,
            label="EWM (21d)" if label == panel_labels[0] else None)

    ax.axhline(60, linestyle="--", linewidth=1, color="red")
    ax.text(
        rhl.index[-1], 62, r"60d threshold",
        ha="right", va="bottom", fontsize=12, color="red"
    )

    ax.fill_between(
        rhl_smooth.index, rhl_smooth.values, 60,
        where=(rhl_smooth.fillna(0).values > 60),
        alpha=0.2, color="red"
    )

    ax.set_ylabel(r"Half-Life (days)")
    ax.set_ylim(bottom=0)

    ax.text(0.02, 0.96, label,
            transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Single shared legend on first axis
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_,
           loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/spread_rolling_halflife.pdf", bbox_inches="tight")
plt.close(fig)


### Rolling z-score

In [27]:
n_pairs = len(cointegrated_pairs)
fig_plotly = make_subplots(
    rows=n_pairs, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r['y']}/{r['x']}" for _, r in cointegrated_pairs.iterrows()],
    vertical_spacing=0.08
)

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    rz = compute_rolling_zscore(spread, window=60)

    fig_plotly.add_trace(
        go.Scatter(x=rz.index, y=rz.values, name=f"{row['y']}/{row['x']}",
                   line=dict(width=1), showlegend=False),
        row=i, col=1
    )
    for level, dash, label in [(2, "dot", "+2σ"), (-2, "dot", "-2σ"),
                                (1, "dash", "+1σ"), (-1, "dash", "-1σ"),
                                (0, "solid", "Mean")]:
        fig_plotly.add_hline(
            y=level, line=dict(color="gray", dash=dash, width=0.8),
            annotation_text=label if i == 1 else None,
            annotation_font_size=9,
            row=i, col=1
        )

fig_plotly.update_yaxes(title_text="Z-score", zeroline=True)
fig_plotly.update_xaxes(title_text="Date", row=n_pairs, col=1)
fig_plotly.update_layout(
    title="Rolling Z-Score (60-day window) — Cointegrated Pairs",
    height=350 * n_pairs,
    template="plotly_white"
)
fig_plotly.show()

In [28]:
n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(10, 4 * n_pairs), sharex=True)
if n_pairs == 1:
    axes = [axes]

panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    rz = compute_rolling_zscore(spread, window=60)

    first = (label == panel_labels[0])

    line, = ax.plot(rz.index, rz.values, linewidth=1,
                label="Z-score" if first else None)
    line_colour = line.get_color()

    ax.axhline( 2, linestyle=":",  linewidth=0.8,
               label=r"$\pm2\sigma$" if first else None)
    ax.axhline(-2, linestyle=":",  linewidth=0.8)
    ax.axhline( 1, linestyle="--", linewidth=0.8,
               label=r"$\pm1\sigma$" if first else None)
    ax.axhline(-1, linestyle="--", linewidth=0.8)
    ax.axhline( 0, linestyle="-",  linewidth=0.6, color="black",
               label="Mean" if first else None)

    ax.fill_between(rz.index,  2, rz.values.clip(min=2), color=line_colour, alpha=0.15)
    ax.fill_between(rz.index, -2, rz.values.clip(max=-2), color=line_colour, alpha=0.15)

    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")
    ax.set_ylabel("Z-score")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/spread_rolling_zscore.pdf", bbox_inches="tight")
plt.close(fig)


## Saving statistics data to CSV

In [30]:
# Save confirmed cointegrated pairs with hedge ratios
cointegrated_pairs.to_csv("../../data/processed/cointegrated_pairs.csv", index=False)

# Save spread summary statistics
summary_df.to_csv("../../data/processed/spread_summary.csv")

# Save the log prices (needed for spread reconstruction in later notebooks)
prices_log.to_csv("../../data/processed/prices_log.csv")